In [ ]:
"""
DocumentProcessor
=================
Converts a folder of research PDFs into a structured JSON file
ready for contextual retrieval evaluation.

Output format (codebase_chunks.json):
[
  {
    "doc_id":        "filename_stem",
    "original_uuid": "uuid4 string",
    "content":       "full raw document text",
    "chunks": [
      {
        "chunk_id":       "doc_id_chunk_0",
        "original_index": 0,
        "content":        "chunk text"
      },
      ...
    ]
  },
  ...
]

Obstacles and how they are handled:
-------------------------------------
1. DIRTY EXTRACTION
   PDFs are layout files. PyMuPDF extracts block by block, producing
   headers, footers, page numbers, and hyphenated line breaks mixed
   into the text.
   → Strip top/bottom 7% of page height (headers/footers)
   → Drop short numeric-only blocks (table noise)
   → Rejoin hyphenated line breaks from two-column layouts
   → Merge lines where next line starts lowercase (mid-sentence break)

2. CROSS-PAGE SENTENCES
   A sentence starting on page 5 and ending on page 6 is split across
   two page extractions.
   → Insert |||PAGE_N||| markers between pages before sentence splitting
   → After spaCy splits sentences, recover page refs from markers
   → Strip markers from final sentence text

3. CHUNK SIZE vs TOPIC COHERENCE
   Fixed character splitting ignores section boundaries — one chunk
   can cover BM25, DPR, and ColBERT, diluting its embedding.
   → Detect section headings via font size / bold flag (no API cost)
   → Split first at hard structural boundaries
   → For blocks still over max_tokens, find topic shift points using
     cosine similarity between sentence windows (all-MiniLM-L6-v2)
   → Recurse until all chunks are within token budget

4. REFERENCES SECTION
   The last 20-40 citation entries are noise for retrieval.
   → Stop extraction when a block's first line matches known headings:
     "references", "bibliography", "works cited"
"""

import json
import logging
import re
import uuid
from collections import Counter
from pathlib import Path

import fitz          # PyMuPDF
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[
        logging.FileHandler("document_processor.log"),
        logging.StreamHandler(),
    ],
)

# ── Module-level singletons (loaded once, reused for all PDFs) ────────────────
_nlp = spacy.load("en_core_web_sm")
_nlp.disable_pipes(["ner", "tagger", "lemmatizer"])

_embedder = SentenceTransformer("all-MiniLM-L6-v2")

# ── Constants ─────────────────────────────────────────────────────────────────
REFERENCE_HEADINGS   = {"references", "bibliography", "works cited"}
HEADING_SIZE_FACTOR  = 1.15   # font must be this much larger than body to be a heading
MIN_BLOCK_SENTENCES  = 3      # blocks smaller than this are merged into their neighbour


# =============================================================================
# DocumentProcessor
# =============================================================================

class DocumentProcessor:
    """
    Converts all PDFs in a folder into a single JSON file.

    Usage:
        processor = DocumentProcessor(
            pdf_folder   = "./RAG_research_paper",
            output_path  = "./data/codebase_chunks.json",
            max_tokens   = 300,   # target chunk size in tokens
            overlap      = 2,     # overlap sentences between adjacent chunks
        )
        processor.run()
    """

    def __init__(
        self,
        pdf_folder:  str,
        output_path: str  = "./data/codebase_chunks.json",
        max_tokens:  int  = 300,
        overlap:     int  = 2,
    ):
        self.pdf_folder  = Path(pdf_folder)
        self.output_path = Path(output_path)
        self.max_tokens  = max_tokens
        self.overlap     = overlap

    # ── Public entry point ────────────────────────────────────────────────────

    def run(self) -> list[dict]:
        """
        Process every PDF in the folder and save results to JSON.
        Returns the dataset list so callers can use it directly.
        """
        pdf_files = sorted(self.pdf_folder.glob("*.pdf"))
        if not pdf_files:
            raise FileNotFoundError(f"No PDFs found in '{self.pdf_folder}'")

        logging.info(f"Found {len(pdf_files)} PDF(s) in '{self.pdf_folder}'")

        dataset      = []
        skipped      = []

        for pdf_file in pdf_files:
            logging.info(f"{'─' * 60}")
            logging.info(f"Processing: {pdf_file.name}")

            result = self._process_single_pdf(str(pdf_file))

            if result is None:
                skipped.append(pdf_file.name)
                continue

            dataset.append(result)
            logging.info(
                f"'{pdf_file.name}' — done. "
                f"{len(result['chunks'])} chunks produced."
            )

        if skipped:
            logging.warning(f"Skipped {len(skipped)} file(s): {skipped}")

        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.output_path, "w", encoding="utf-8") as f:
            json.dump(dataset, f, indent=2, ensure_ascii=False)

        logging.info(f"{'═' * 60}")
        logging.info(
            f"Saved {len(dataset)} document(s) → '{self.output_path}'"
        )

        return dataset

    # ── Per-file orchestration ─────────────────────────────────────────────────

    def _process_single_pdf(self, pdf_path: str) -> dict | None:
        """
        Full pipeline for one PDF:
          1. Extract text page by page (obstacle 1 + 4)
          2. Build raw_document_text before any chunking (clean source for Claude context)
          3. Split into sentences with page markers (obstacle 2)
          4. Detect structural boundaries via font metadata
          5. Build coarse blocks at section headings
          6. Semantically refine oversized blocks (obstacle 3)
          7. Add sentence overlap between adjacent chunks
          8. Format into the JSON schema
        """
        # Step 1 — Extract
        pages = self._extract_pages(pdf_path)
        if not pages:
            logging.error(f"'{pdf_path}' — no extractable text.")
            return None

        # Step 2 — Raw document text (BEFORE chunking, no markers, no overlap)
        raw_document_text = " ".join(p["text"] for p in pages)

        # Step 3 — Sentence splitting with page markers
        sentences, page_refs = self._split_sentences_with_pages(pages)
        if not sentences:
            logging.error(f"'{pdf_path}' — no sentences after splitting.")
            return None

        # Step 4 — Structural boundaries
        hard_boundaries = self._detect_heading_boundaries(pdf_path, sentences)

        # Step 5 — Coarse blocks
        blocks = self._build_coarse_blocks(sentences, page_refs, hard_boundaries)

        logging.info(
            f"'{pdf_path}' — {len(blocks)} coarse blocks, "
            f"{sum(1 for b in blocks if b['needs_split'])} need semantic refinement."
        )

        # Step 6 — Semantic refinement of oversized blocks
        refined = []
        for block in blocks:
            if block["needs_split"]:
                sub = self._semantic_split(block)
                for s in sub:
                    s["was_semantic"] = True
                refined.extend(sub)
            else:
                block["was_semantic"] = False
                refined.append(block)

        # Step 7 — Overlap + final chunk format
        chunks = self._finalize_chunks(refined)

        # Step 8 — Build JSON record
        doc_id   = Path(pdf_path).stem
        doc_uuid = str(uuid.uuid4())

        return {
            "doc_id":        doc_id,
            "original_uuid": doc_uuid,
            "content":       raw_document_text,
            "chunks": [
                {
                    "chunk_id":       f"{doc_id}_chunk_{i}",
                    "original_index": i,
                    "content":        c["text"],
                }
                for i, c in enumerate(chunks)
            ],
        }

    # =========================================================================
    # STEP 1 — Page extraction
    # OBSTACLE: headers, footers, table noise, hyphenated line breaks,
    #           mid-sentence newlines, and the references section.
    # =========================================================================

    def _extract_pages(self, pdf_path: str) -> list[dict] | None:
        doc  = fitz.open(pdf_path)
        pages = []
        references_reached = False

        for i, page in enumerate(doc):
            if references_reached:
                break

            page_height = page.rect.height
            blocks = page.get_text("blocks")
            lines  = []

            for block in blocks:
                text   = block[4].strip()
                y_top  = block[1]
                y_bot  = block[3]

                # Obstacle 4 — stop at references heading
                first_line = text.split("\n")[0].strip().lower()
                if first_line in REFERENCE_HEADINGS:
                    logging.info(
                        f"'{pdf_path}' — references section at page {i + 1}, stopping."
                    )
                    references_reached = True
                    break

                # Obstacle 1a — strip headers and footers
                if y_top < page_height * 0.07:
                    continue
                if y_bot  > page_height * 0.93:
                    continue

                # Obstacle 1b — drop short numeric blocks (table noise)
                if re.fullmatch(r"[\d\s\.\,\%\-]+", text) and len(text) < 40:
                    continue

                lines.extend(text.split("\n"))

            if references_reached:
                break

            # Obstacle 1c — rejoin hyphenated line breaks
            rejoined = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                if rejoined and rejoined[-1].endswith("-"):
                    rejoined[-1] = rejoined[-1][:-1] + line
                else:
                    rejoined.append(line)

            # Obstacle 1d — merge lines where next starts lowercase
            cleaned = []
            for line in rejoined:
                if (
                    cleaned
                    and not cleaned[-1][-1] in ".!?"
                    and line
                    and line[0].islower()
                ):
                    cleaned[-1] += " " + line
                else:
                    cleaned.append(line)

            text = " ".join(cleaned).strip()
            if text:
                pages.append({"page_num": i + 1, "text": text})

        doc.close()
        return pages if pages else None

    # =========================================================================
    # STEP 2 — Sentence splitting with page markers
    # OBSTACLE: sentences that span two pages would be split without markers.
    # =========================================================================

    def _split_sentences_with_pages(
        self, pages: list[dict]
    ) -> tuple[list[str], list[int]]:
        # Insert |||PAGE_N||| markers between pages so spaCy can split
        # across them — markers are recovered and stripped afterwards.
        parts = []
        for p in pages:
            parts.append(p["text"])
            parts.append(f"|||PAGE_{p['page_num'] + 1}|||")

        joined         = " ".join(parts)
        max_sent_chars = self.max_tokens * 4

        # spaCy sentence splitting — called ONCE for the whole document
        raw_sentences = self._spacy_split(joined, max_sent_chars)

        sentences:    list[str] = []
        page_refs:    list[int] = []
        current_page: int       = pages[0]["page_num"]
        marker_re                = re.compile(r"\|\|\|PAGE_(\d+)\|\|\|")

        for sent in raw_sentences:
            match = marker_re.search(sent)
            if match:
                page_refs.append(current_page)
                current_page = int(match.group(1))
                clean = marker_re.sub("", sent).strip()
                if clean and re.search(r"[a-zA-Z0-9]", clean):
                    sentences.append(clean)
                else:
                    page_refs.pop()
            else:
                sentences.append(sent)
                page_refs.append(current_page)

        return sentences, page_refs

    def _spacy_split(self, text: str, max_chars: int) -> list[str]:
        doc = _nlp(text)
        out = []
        for sent in doc.sents:
            s = sent.text.strip()
            if not s or not re.search(r"[a-zA-Z0-9]", s):
                continue
            if len(s) > max_chars:
                # Split oversized sentence at nearest space to midpoint
                mid = len(s) // 2
                cut = s.rfind(" ", 0, mid) or s.find(" ", mid)
                if cut != -1:
                    out.append(s[:cut].strip())
                    out.append(s[cut:].strip())
                    continue
            out.append(s)
        return out

    # =========================================================================
    # STEP 3 — Structural boundary detection (font scan, zero API cost)
    # OBSTACLE: without section boundaries, blocks span multiple topics.
    # =========================================================================

    def _detect_heading_boundaries(
        self, pdf_path: str, sentences: list[str]
    ) -> set[int]:
        doc = fitz.open(pdf_path)

        # Pass 1 — find body font size (most common font size in document)
        sizes = []
        for page in doc:
            h = page.rect.height
            for block in page.get_text("dict")["blocks"]:
                if block.get("type") != 0:
                    continue
                if block["bbox"][1] < h * 0.07 or block["bbox"][3] > h * 0.93:
                    continue
                for line in block.get("lines", []):
                    for span in line.get("spans", []):
                        if span["text"].strip():
                            sizes.append(round(span["size"], 1))

        if not sizes:
            doc.close()
            return set()

        body_size = Counter(sizes).most_common(1)[0][0]

        # Pass 2 — collect heading candidates
        heading_pages: dict[str, set[int]] = {}

        for page_num, page in enumerate(doc):
            h = page.rect.height
            for block in page.get_text("dict")["blocks"]:
                if block.get("type") != 0:
                    continue
                if block["bbox"][1] < h * 0.07 or block["bbox"][3] > h * 0.93:
                    continue

                span_texts   = []
                is_candidate = False

                for line in block.get("lines", []):
                    for span in line.get("spans", []):
                        t = span["text"].strip()
                        if not t:
                            continue
                        span_texts.append(t)
                        is_bold    = bool(span["flags"] & 16)
                        is_larger  = round(span["size"], 1) > body_size * HEADING_SIZE_FACTOR
                        if is_bold or is_larger:
                            is_candidate = True

                if not is_candidate:
                    continue

                block_text = " ".join(span_texts).strip()
                if not block_text or not re.search(r"[a-zA-Z]{2,}", block_text):
                    continue

                key = block_text.lower().strip()
                if key in REFERENCE_HEADINGS:
                    continue

                heading_pages.setdefault(key, set()).add(page_num)

        doc.close()

        # Headings on 3+ pages are running headers — discard
        true_headings = {
            k for k, pages in heading_pages.items() if len(pages) < 3
        }

        if not true_headings:
            return set()

        # Map headings to sentence indices
        boundaries: set[int] = set()
        for i, sent in enumerate(sentences):
            sl = sent.lower().strip()
            for h in true_headings:
                if h in sl or sl in h:
                    boundaries.add(i)
                    break

        logging.info(f"  {len(boundaries)} structural boundaries detected.")
        return boundaries

    # =========================================================================
    # STEP 4 — Build coarse blocks at hard boundaries
    # =========================================================================

    def _estimate_tokens(self, sentences: list[str]) -> int:
        total = 0
        for s in sentences:
            ascii_c     = sum(1 for c in s if ord(c) < 128)
            non_ascii_c = len(s) - ascii_c
            total += (ascii_c // 4) + (non_ascii_c // 2)
        return total

    def _build_coarse_blocks(
        self,
        sentences:       list[str],
        page_refs:       list[int],
        hard_boundaries: set[int],
    ) -> list[dict]:
        blocks: list[dict] = []
        cur_sents: list[str] = []
        cur_pages: list[int] = []

        def flush():
            if cur_sents:
                tok = self._estimate_tokens(cur_sents)
                blocks.append({
                    "sentences":   cur_sents[:],
                    "page_refs":   cur_pages[:],
                    "token_est":   tok,
                    "needs_split": tok > self.max_tokens,
                })

        for i, (sent, page) in enumerate(zip(sentences, page_refs)):
            if i in hard_boundaries and cur_sents:
                flush()
                cur_sents, cur_pages = [], []
            cur_sents.append(sent)
            cur_pages.append(page)

        flush()

        # Merge tiny blocks into their predecessor
        merged: list[dict] = []
        for block in blocks:
            if len(block["sentences"]) < MIN_BLOCK_SENTENCES and merged:
                merged[-1]["sentences"].extend(block["sentences"])
                merged[-1]["page_refs"].extend(block["page_refs"])
                merged[-1]["token_est"]   = self._estimate_tokens(merged[-1]["sentences"])
                merged[-1]["needs_split"] = merged[-1]["token_est"] > self.max_tokens
            else:
                merged.append(block)

        return merged

    # =========================================================================
    # STEP 5 — Semantic refinement of oversized blocks
    # OBSTACLE: a block covering multiple topics needs to be split at the
    #           point where the topic actually shifts, not at a fixed position.
    # =========================================================================

    def _semantic_split(
        self,
        block:     dict,
        window:    int   = 5,
        threshold: float = 0.15,
    ) -> list[dict]:
        sents = block["sentences"]
        pages = block["page_refs"]

        if len(sents) < window * 2:
            return [block]

        windows = [
            " ".join(sents[i: i + window])
            for i in range(0, len(sents), window)
        ]
        if len(windows) < 2:
            return [block]

        embeddings = _embedder.encode(windows, convert_to_numpy=True,show_progress_bar=False)

        similarities = [
            float(
                np.dot(embeddings[i], embeddings[i + 1])
                / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1]) + 1e-8)
            )
            for i in range(len(embeddings) - 1)
        ]

        # Find threshold-based candidates (local minima with significant drop)
        candidates = []
        for i in range(1, len(similarities) - 1):
            is_local_min = (
                similarities[i] < similarities[i - 1]
                and similarities[i] < similarities[i + 1]
            )
            if not is_local_min:
                continue
            drop = min(similarities[i - 1], similarities[i + 1]) - similarities[i]
            if drop > threshold:
                candidates.append(((i + 1) * window, drop))

        # FIX: if no threshold candidate found, fall back to the minimum
        # similarity point — guarantees every oversized block gets split
        if not candidates:
            min_idx   = int(np.argmin(similarities))
            split_at  = (min_idx + 1) * window
            # Make sure split_at is within bounds
            split_at  = max(window, min(split_at, len(sents) - window))
            candidates = [(split_at, 0.0)]

        midpoint = len(sents) // 2
        candidates.sort(key=lambda x: (abs(x[0] - midpoint), -x[1]))
        split_at = candidates[0][0]

        def make_block(s, p):
            tok = self._estimate_tokens(s)
            return {
                "sentences":   s,
                "page_refs":   p,
                "token_est":   tok,
                "needs_split": tok > self.max_tokens,
            }

        left  = make_block(sents[:split_at], pages[:split_at])
        right = make_block(sents[split_at:], pages[split_at:])

        result = []
        for half in (left, right):
            if half["needs_split"]:
                result.extend(self._semantic_split(half, window, threshold))
            else:
                result.append(half)

        return result

        # =========================================================================
        # STEP 6 — Finalize chunks with sentence overlap
        # Overlap lets adjacent chunks share a few sentences so context is not
        # lost at chunk boundaries — important for questions that span sections.
        # =========================================================================

    def _finalize_chunks(self, blocks: list[dict]) -> list[dict]:
        chunks = []

        for i, block in enumerate(blocks):
            own_sents = block["sentences"]
            own_pages = block["page_refs"]

            if i > 0 and self.overlap > 0:
                prev       = blocks[i - 1]
                ol_sents   = prev["sentences"][-self.overlap:]
                ol_pages   = prev["page_refs"][-self.overlap:]
                full_sents = ol_sents + own_sents
                full_pages = ol_pages + own_pages
            else:
                full_sents = own_sents
                full_pages = own_pages

            chunks.append({
                "text":       " ".join(full_sents),
                "page_start": own_pages[0],
                "page_end":   full_pages[-1],
            })

        return chunks


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":
    processor = DocumentProcessor(
        pdf_folder  = "./RAG_research_paper",
        output_path = "./data/codebase_chunks.json",
        max_tokens  = 256,   # smaller = more focused chunks = better retrieval
        overlap     = 2,
    )

    dataset = processor.run()

    # Quick sanity check
    for doc in dataset:
        print(f"\n{doc['doc_id']}")
        print(f"  UUID:   {doc['original_uuid']}")
        print(f"  Chunks: {len(doc['chunks'])}")
        print(f"  First chunk preview: {doc['chunks'][0]['content'][:120]}...")

2026-03-04 21:25:18,416 — INFO — Use pytorch device_name: cpu
2026-03-04 21:25:18,416 — INFO — Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-03-04 21:25:19,630 — INFO — Found 1 PDF(s) in 'RAG_research_paper'
2026-03-04 21:25:19,631 — INFO — ────────────────────────────────────────────────────────────
2026-03-04 21:25:19,631 — INFO — Processing: 2312.10997v5.pdf
2026-03-04 21:25:19,681 — INFO — 'RAG_research_paper\2312.10997v5.pdf' — references section at page 17, stopping.
2026-03-04 21:25:21,168 — INFO —   8 structural boundaries detected.
2026-03-04 21:25:21,172 — INFO — 'RAG_research_paper\2312.10997v5.pdf' — 7 coarse blocks, 5 need semantic refinement.
Batches: 100%|██████████| 1/1 [00:00<00:00, 22.73it/s]
2026-03-04 21:25:28,538 — INFO — '2312.10997v5.pdf' — done. 69 chunks produced.
2026-03-04 21:25:28,543 — INFO — ════════════════════════════════════════════════════════════
2026-03-04 21:25:28,544 — INFO — Saved 1 document(s) → 'data\codebase_chunks.json'



2312.10997v5
  UUID:   6b66b9c2-fe88-49d8-a003-02f0b892c7f2
  Chunks: 69
  First chunk preview: Retrieval-Augmented Generation for Large Language Models: A Survey Yunfan Gaoa, Yun Xiongb, Xinyu Gaob, Kangxiang Jiab, ...
